# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tahaahmed729/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Method Choice and Why
- **Task Type:** Binary Classification / Tabular Prediction
- **Chosen Model:** Random Forest & Logistic Regression (Evaluated against Naive Baseline)
- **Justification:** Tabular signals perform best with tree-based ensembles without assuming linearity. Random Forest captures non-linear interactions without heavy parameter tuning while resisting overfitting.
- **Complexity Trade-off:** Starting with Logistic Regression (interpretable linear boundary) before escalating to Random Forest to verify if non-linear complexity actually yields performance gains.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold

# 1. Load Data
df = pd.read_parquet('../data/processed_signals.parquet') # Apni file path adjust karo

target_col = 'target'
features = [c for c in df.columns if c not in [target_col, 'group_id']]

# 2. Strict Split (Ensuring exact match with Week 4 split strategy)
# GroupKFold or simple train/test split based on your week 4 design:
X = df[features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class ratio:\n{y_train.value_counts(normalize=True)}")


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# 1. Week 4 Baseline Model (Majority Class / Heuristic)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
y_prob_base = baseline.predict_proba(X_test)[:, 1]

# 2. Linear Candidate
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

# 3. Tree-based Candidate
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# 4. Model vs Baseline Table
metrics_summary = pd.DataFrame([
    {
        "Model": "Week 4 Baseline (Dummy)",
        "ROC-AUC": roc_auc_score(y_test, y_prob_base),
        "F1-Score": f1_score(y_test, y_pred_base, zero_division=0),
        "Precision": precision_score(y_test, y_pred_base, zero_division=0),
        "Recall": recall_score(y_test, y_pred_base, zero_division=0)
    },
    {
        "Model": "Logistic Regression",
        "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
        "F1-Score": f1_score(y_test, y_pred_lr, zero_division=0),
        "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
        "Recall": recall_score(y_test, y_pred_lr, zero_division=0)
    },
    {
        "Model": "Random Forest (Chosen)",
        "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
        "F1-Score": f1_score(y_test, y_pred_rf, zero_division=0),
        "Precision": precision_score(y_test, y_pred_rf, zero_division=0),
        "Recall": recall_score(y_test, y_pred_rf, zero_division=0)
    }
])

display(metrics_summary)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

# 1. Permutation Importance
perm_importance = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
sorted_idx = perm_importance.importances_mean.argsort()[-10:]

plt.figure(figsize=(8, 4))
plt.barh(X.columns[sorted_idx], perm_importance.importances_mean[sorted_idx])
plt.xlabel("Permutation Importance (Mean Decrease in Score)")
plt.title("Top 10 Feature Importances on Validation Set")
plt.tight_layout()
plt.show()

# 2. Error Slicing
test_df = X_test.copy()
test_df['actual'] = y_test
test_df['predicted'] = y_pred_rf
test_df['prob'] = y_prob_rf
test_df['is_error'] = test_df['actual'] != test_df['predicted']

print("Top Error Slices Summary:")
print(test_df.groupby('is_error')[features[:3]].mean())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.